# JETANK Notebook Mobile/Browser Control with Camera

Run this notebook in the same Jupyter environment where the official JETANK notebooks work.

It provides:

- tracked-base movement buttons
- jog controls for servos 1-5
- a dedicated camera tilt section for servo 5
- live camera preview
- photo capture to `./filmed_stuff`
- video recording to `./filmed_stuff`

Safety: lift the tracks off the ground for the first test, keep the base speed low, and press **STOP ALL** before closing the notebook.

In [ ]:
import os
import time
import threading
from pathlib import Path

import cv2
import ipywidgets.widgets as widgets
import traitlets
from IPython.display import display

from jetbot import Robot, Camera, bgr8_to_jpeg
from SCSCtrl import TTLServo

MEDIA_DIR = Path('./filmed_stuff')
MEDIA_DIR.mkdir(parents=True, exist_ok=True)

print('Media directory:', MEDIA_DIR.resolve())
print('Imports OK. This notebook is using the same style as the official JETANK notebooks.')

## Initialize hardware

This creates the robot and camera objects. If the camera fails, restart the notebook kernel and make sure no other notebook is still using the camera.

In [ ]:
CAMERA_WIDTH = 300
CAMERA_HEIGHT = 300
CAMERA_FPS = 15

robot = Robot()
robot.stop()

camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS)

image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH, height=CAMERA_HEIGHT, layout=widgets.Layout(width='100%', max_width='360px'))
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

display(image_widget)
print('Hardware initialized. Base is stopped.')

## Shared control state

The servo target angles below are software-tracked estimates. If a servo was moved manually or by another notebook, click **Set** or **Center** to resynchronize the physical pose.

In [ ]:
SERVO_LIMITS = {
    1: {'name': 'Servo 1 / pan or arm base', 'min': -80, 'max': 80, 'angle': 0},
    2: {'name': 'Servo 2 / shoulder', 'min': -90, 'max': 90, 'angle': 0},
    3: {'name': 'Servo 3 / elbow', 'min': -90, 'max': 90, 'angle': 0},
    4: {'name': 'Servo 4 / gripper', 'min': -90, 'max': 0, 'angle': -45},
    5: {'name': 'Servo 5 / camera tilt', 'min': -45, 'max': 25, 'angle': 0},
}

servo_lock = threading.RLock()
base_lock = threading.RLock()
record_lock = threading.RLock()

recording = False
record_thread = None
record_writer = None
record_path = None

def clamp(value, low, high):
    return max(low, min(high, value))

def now_tag():
    return time.strftime('%Y%m%d_%H%M%S')

def safe_status(message):
    status.value = str(message)

print('Control state ready.')

## Movement and servo functions

In [ ]:
def move_base(direction):
    speed = float(base_speed.value)
    with base_lock:
        if direction == 'forward':
            robot.forward(speed)
        elif direction == 'backward':
            robot.backward(speed)
        elif direction == 'left':
            robot.left(speed)
        elif direction == 'right':
            robot.right(speed)
        elif direction == 'stop':
            robot.stop()
        else:
            raise ValueError('Unknown base direction: {}'.format(direction))
    safe_status('Base: {}'.format(direction))


def stop_base(_=None):
    with base_lock:
        robot.stop()
    safe_status('Base stopped')


def jog_servo(servo_id, delta):
    servo_id = int(servo_id)
    delta = float(delta)
    cfg = SERVO_LIMITS[servo_id]
    with servo_lock:
        target = clamp(float(cfg['angle']) + delta, float(cfg['min']), float(cfg['max']))
        cfg['angle'] = target
        TTLServo.servoAngleCtrl(servo_id, target, 1, int(servo_speed.value))
        time.sleep(float(servo_stop_delay.value))
        TTLServo.servoStop(servo_id)
    angle_labels[servo_id].value = '{:.1f} deg'.format(target)
    safe_status('Servo {} -> {:.1f} deg'.format(servo_id, target))


def set_servo(servo_id, angle):
    servo_id = int(servo_id)
    cfg = SERVO_LIMITS[servo_id]
    with servo_lock:
        target = clamp(float(angle), float(cfg['min']), float(cfg['max']))
        cfg['angle'] = target
        TTLServo.servoAngleCtrl(servo_id, target, 1, int(servo_speed.value))
    angle_labels[servo_id].value = '{:.1f} deg'.format(target)
    safe_status('Servo {} set to {:.1f} deg'.format(servo_id, target))


def stop_servo(servo_id):
    servo_id = int(servo_id)
    with servo_lock:
        TTLServo.servoStop(servo_id)
    safe_status('Servo {} stopped'.format(servo_id))


def stop_all(_=None):
    stop_base()
    with servo_lock:
        for servo_id in sorted(SERVO_LIMITS):
            try:
                TTLServo.servoStop(servo_id)
                time.sleep(0.002)
            except Exception as exc:
                print('Failed to stop servo {}: {}'.format(servo_id, exc))
    safe_status('STOP ALL complete')

print('Movement functions ready.')

## Camera capture and recording functions

Photos and videos are saved into `./filmed_stuff` relative to this notebook's running directory.

In [ ]:
def latest_frame():
    frame = camera.value
    if frame is None:
        raise RuntimeError('Camera returned no frame')
    return frame.copy()


def take_photo(_=None):
    frame = latest_frame()
    path = MEDIA_DIR / 'photo_{}.jpg'.format(now_tag())
    ok = cv2.imwrite(str(path), frame)
    if not ok:
        raise RuntimeError('Failed to save photo: {}'.format(path))
    last_photo.value = bgr8_to_jpeg(frame)
    safe_status('Photo saved: {}'.format(path.resolve()))


def _record_loop(fps):
    global recording, record_writer
    delay = 1.0 / max(1, int(fps))
    while True:
        with record_lock:
            if not recording or record_writer is None:
                return
        try:
            frame = latest_frame()
            if frame.shape[1] != CAMERA_WIDTH or frame.shape[0] != CAMERA_HEIGHT:
                frame = cv2.resize(frame, (CAMERA_WIDTH, CAMERA_HEIGHT))
            with record_lock:
                if recording and record_writer is not None:
                    record_writer.write(frame)
        except Exception as exc:
            print('Recording warning:', exc)
        time.sleep(delay)


def start_recording(_=None):
    global recording, record_thread, record_writer, record_path
    with record_lock:
        if recording:
            safe_status('Already recording: {}'.format(record_path))
            return
        record_path = MEDIA_DIR / 'record_{}.mp4'.format(now_tag())
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        record_writer = cv2.VideoWriter(str(record_path), fourcc, float(record_fps.value), (CAMERA_WIDTH, CAMERA_HEIGHT))
        if not record_writer.isOpened():
            record_writer = None
            raise RuntimeError('Failed to open video writer: {}'.format(record_path))
        recording = True
        record_thread = threading.Thread(target=_record_loop, args=(int(record_fps.value),))
        record_thread.daemon = True
        record_thread.start()
    record_indicator.value = 'Recording...'
    safe_status('Recording started: {}'.format(record_path.resolve()))


def stop_recording(_=None):
    global recording, record_thread, record_writer, record_path
    with record_lock:
        if not recording:
            safe_status('Not recording')
            return
        saved_path = record_path
        recording = False
        thread = record_thread
    if thread is not None and thread.is_alive():
        thread.join(timeout=2.0)
    with record_lock:
        if record_writer is not None:
            record_writer.release()
        record_writer = None
        record_thread = None
        record_path = None
    record_indicator.value = 'Idle'
    safe_status('Recording saved: {}'.format(saved_path.resolve()))


def shutdown_camera(_=None):
    try:
        stop_recording()
    except Exception as exc:
        print('Recording stop warning:', exc)
    try:
        camera_link.unlink()
    except Exception:
        pass
    try:
        camera.stop()
    except Exception as exc:
        print('Camera stop warning:', exc)
    safe_status('Camera stopped')

print('Camera functions ready.')

## Control UI

For the base, click a direction to start moving and click **Stop Base** to stop. Keep speed low for the first test.

In [ ]:
from IPython.display import HTML

display(HTML("""
<style>
.jp-OutputArea-output { overflow-x: visible !important; }
.widget-box, .jupyter-widgets.widget-box { max-width: 100% !important; }
.widget-button { font-size: 14px !important; }
</style>
"""))

# Ultra-narrow mobile-first layout for JupyterLab on phones.
# JupyterLab itself consumes a lot of horizontal space, so avoid almost all multi-column rows.
full_button_layout = widgets.Layout(width='220px', max_width='220px', height='52px')
small_button_layout = widgets.Layout(width='46px', max_width='46px', height='38px')
angle_layout = widgets.Layout(width='74px', max_width='74px', height='32px')
row_layout = widgets.Layout(width='230px', max_width='230px', justify_content='center', align_items='center')
panel_layout = widgets.Layout(width='230px', max_width='230px')
slider_layout = widgets.Layout(width='230px', max_width='230px')

status = widgets.HTML(value='Ready', layout=widgets.Layout(width='230px', max_width='230px'))

base_speed = widgets.FloatSlider(value=0.25, min=0.0, max=0.6, step=0.01, description='Speed', readout_format='.2f', layout=slider_layout)
servo_step = widgets.FloatSlider(value=2.0, min=1.0, max=15.0, step=1.0, description='Step', layout=slider_layout)
servo_speed = widgets.IntSlider(value=120, min=20, max=300, step=5, description='Servo spd', layout=slider_layout)
servo_stop_delay = widgets.FloatSlider(value=0.18, min=0.0, max=0.5, step=0.01, description='Delay', layout=slider_layout)
record_fps = widgets.IntSlider(value=10, min=2, max=20, step=1, description='FPS', layout=slider_layout)

forward_button = widgets.Button(description='Forward', layout=full_button_layout)
backward_button = widgets.Button(description='Backward', layout=full_button_layout)
left_button = widgets.Button(description='Left', layout=full_button_layout)
right_button = widgets.Button(description='Right', layout=full_button_layout)
stop_base_button = widgets.Button(description='Stop Base', button_style='danger', layout=full_button_layout)
stop_all_button = widgets.Button(description='STOP ALL', button_style='danger', layout=full_button_layout)

forward_button.on_click(lambda _: move_base('forward'))
backward_button.on_click(lambda _: move_base('backward'))
left_button.on_click(lambda _: move_base('left'))
right_button.on_click(lambda _: move_base('right'))
stop_base_button.on_click(stop_base)
stop_all_button.on_click(stop_all)

base_pad = widgets.VBox([
    forward_button,
    left_button,
    right_button,
    backward_button,
    stop_base_button,
], layout=panel_layout)

angle_labels = {}
servo_rows = []
for servo_id in sorted(SERVO_LIMITS):
    cfg = SERVO_LIMITS[servo_id]
    minus = widgets.Button(description='-', layout=small_button_layout)
    plus = widgets.Button(description='+', layout=small_button_layout)
    center = widgets.Button(description='Set 0', layout=full_button_layout)
    stop = widgets.Button(description='Stop', layout=full_button_layout)
    label = widgets.HTML('<b>{}</b>'.format(cfg['name']), layout=widgets.Layout(width='230px', max_width='230px'))
    angle_labels[servo_id] = widgets.Label('{:.1f} deg'.format(float(cfg['angle'])), layout=angle_layout)

    minus.on_click(lambda _, sid=servo_id: jog_servo(sid, -float(servo_step.value)))
    plus.on_click(lambda _, sid=servo_id: jog_servo(sid, float(servo_step.value)))
    center.on_click(lambda _, sid=servo_id: set_servo(sid, 0))
    stop.on_click(lambda _, sid=servo_id: stop_servo(sid))

    servo_rows.append(widgets.VBox([
        label,
        widgets.HBox([minus, angle_labels[servo_id], plus], layout=row_layout),
        center,
        stop,
    ], layout=widgets.Layout(width='230px', max_width='230px', margin='6px 0 12px 0')))

camera_up = widgets.Button(description='Camera Up', layout=full_button_layout)
camera_down = widgets.Button(description='Camera Down', layout=full_button_layout)
camera_center = widgets.Button(description='Camera Center', layout=full_button_layout)
stop_tilt = widgets.Button(description='Stop Camera Tilt', layout=full_button_layout)

camera_up.on_click(lambda _: jog_servo(5, -float(servo_step.value)))
camera_down.on_click(lambda _: jog_servo(5, float(servo_step.value)))
camera_center.on_click(lambda _: set_servo(5, 0))
stop_tilt.on_click(lambda _: stop_servo(5))

photo_button = widgets.Button(description='Take Photo', button_style='success', layout=full_button_layout)
start_record_button = widgets.Button(description='Start Recording', button_style='warning', layout=full_button_layout)
stop_record_button = widgets.Button(description='Stop Recording', button_style='danger', layout=full_button_layout)
shutdown_camera_button = widgets.Button(description='Stop Camera', button_style='danger', layout=full_button_layout)
record_indicator = widgets.Label('Idle', layout=widgets.Layout(width='150px'))
last_photo = widgets.Image(format='jpeg', width=160, height=160, layout=widgets.Layout(width='160px', height='160px', max_width='160px'))

photo_button.on_click(take_photo)
start_record_button.on_click(start_recording)
stop_record_button.on_click(stop_recording)
shutdown_camera_button.on_click(shutdown_camera)

ui = widgets.VBox([
    widgets.HTML('<h3>Tracked Base</h3>', layout=panel_layout),
    base_speed,
    base_pad,
    widgets.HTML('<h3>Camera Tilt / Servo 5</h3>', layout=panel_layout),
    camera_up,
    camera_down,
    camera_center,
    stop_tilt,
    widgets.HTML('<h3>All Servos</h3>', layout=panel_layout),
    servo_step,
    servo_speed,
    servo_stop_delay,
    widgets.VBox(servo_rows, layout=panel_layout),
    stop_all_button,
    widgets.HTML('<h3>Capture</h3>', layout=panel_layout),
    record_fps,
    photo_button,
    start_record_button,
    stop_record_button,
    shutdown_camera_button,
    widgets.HBox([widgets.Label('Recording:'), record_indicator], layout=row_layout),
    widgets.HTML('<b>Last photo preview</b>', layout=panel_layout),
    last_photo,
    widgets.HTML('<h3>Status</h3>', layout=panel_layout),
    status,
], layout=panel_layout)

display(ui)


## Cleanup cell

Run this before closing the notebook or switching to another camera/motor notebook.

In [ ]:
stop_all()
shutdown_camera()
print('Cleanup complete.')